# Data tour: what is in the merged tables?

This notebook walks through `data/processed/nakuru_daily.parquet` and `busia_daily.parquet` so you can see, in plots and numbers, what the models will learn from. Each section ends with **what to take away**, because several of those answers decide how we build the models.

**How to run:** open this file in VS Code, choose the `.venv` interpreter as the kernel, and use *Run All*. It reads the processed files, so run `python -m src.features.merge` first if they are missing. The slowest cell takes under a minute.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "config" / "sites.yaml").exists():
            return p
    raise FileNotFoundError("Open this notebook from inside the repo folder.")


ROOT = find_root()
SITES = ["nakuru", "busia"]
COLORS = {"nakuru": "tab:blue", "busia": "tab:orange"}
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

df = {s: pd.read_parquet(ROOT / "data" / "processed" / f"{s}_daily.parquet") for s in SITES}
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

for s, d in df.items():
    print(f"{s:8s} {d.shape[0]:>10,} rows | {d['pixel_id'].nunique()} pixels | "
          f"{d['date'].min():%Y-%m-%d} to {d['date'].max():%Y-%m-%d}")

## 1. What one row looks like

One row is **one pixel on one day**. The first block is a single row turned on its side so you can read every column. The units are in the column names: `precip_mm` is millimetres of rain that day, `rh_pct` is mean relative humidity, `t2m_c` mean air temperature, `sp_hpa` surface pressure, `wind_ms` mean wind speed, `tcc` cloud cover from 0 to 1. The `elev_*` columns never change for a pixel.

In [ ]:
d = df["nakuru"]
display(d.iloc[1000].to_frame("one row (Nakuru)"))
display(d.dtypes.astype(str).to_frame("dtype").T)

## 2. One pixel's story

The pixel nearest to each town, for one year (2020). Watch how different the three panels are: rainfall is spiky and mostly zero, humidity moves in slow waves, temperature is smooth. That difference in "memory" is why some of these variables will be much easier to forecast than others.

In [ ]:
ANCHORS = {"nakuru": (-0.30, 36.07), "busia": (0.46, 34.11)}  # approximate town centres


def nearest_pixel(d, lat, lon):
    px = d[["pixel_id", "lat", "lon"]].drop_duplicates()
    i = ((px["lat"] - lat) ** 2 + (px["lon"] - lon) ** 2).idxmin()
    return px.loc[i, "pixel_id"]


for s in SITES:
    d = df[s]
    pid = nearest_pixel(d, *ANCHORS[s])
    p = d[(d["pixel_id"] == pid) & (d["date"].dt.year == 2020)].set_index("date")
    elev = p["elev_mean_m"].iloc[0]
    fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    ax[0].bar(p.index, p["precip_mm"], color="tab:blue", width=1.0)
    ax[0].set_ylabel("rain (mm/day)")
    ax[1].plot(p.index, p["rh_pct"], color="tab:green")
    ax[1].fill_between(p.index, p["rh_min_pct"], p["rh_max_pct"], color="tab:green", alpha=0.2)
    ax[1].set_ylabel("humidity (%)\nmean, min-max band")
    ax[2].plot(p.index, p["t2m_c"], color="tab:red")
    ax[2].fill_between(p.index, p["t2m_min_c"], p["t2m_max_c"], color="tab:red", alpha=0.2)
    ax[2].set_ylabel("temperature (C)\nmean, min-max band")
    fig.suptitle(f"{s.title()}: pixel {pid}, elevation {elev:.0f} m, year 2020")
    plt.tight_layout()
    plt.show()

**Take away:** rainfall on any one day is hard to predict from the day before. Humidity and temperature persist, so they should be easier. Section 8 puts numbers on this.

## 3. The seasonal cycle

Average monthly rainfall for each site (region-wide average, then averaged over years), with the driest and wettest year as a band, next to average humidity. Kenya's two rainy seasons (long rains around March to May, short rains around October to December) should show up as two humps, at least at Nakuru.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
summary = {}
for s in SITES:
    d = df[s]
    regional = d.groupby("date").agg(rain=("precip_mm", "mean"), rh=("rh_pct", "mean"))
    monthly_rain = regional["rain"].resample("MS").sum()          # mm per month
    by_m = monthly_rain.groupby(monthly_rain.index.month).agg(["min", "mean", "max"])
    rh_m = regional["rh"].groupby(regional.index.month).mean()
    ax[0].plot(MONTHS, by_m["mean"], marker="o", color=COLORS[s], label=s)
    ax[0].fill_between(MONTHS, by_m["min"], by_m["max"], color=COLORS[s], alpha=0.15)
    ax[1].plot(MONTHS, rh_m.values, marker="o", color=COLORS[s], label=s)
    summary[s] = by_m["mean"].round(0).values   # .values: the index is month numbers, our labels are names
ax[0].set_title("Monthly rainfall (mm): mean, with driest-to-wettest year band")
ax[1].set_title("Mean relative humidity (%)")
for a in ax:
    a.legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame(summary, index=MONTHS).T)

**Take away:** a model must know the time of year. That is what the sine/cosine day-of-year features are for. If a site shows a clear rainy season, "climatology" (the average for that day of year) will be a strong baseline that the models have to beat.

## 4. Maps: where does it rain, and how high is the ground?

Each square is one pixel. Left: average annual rainfall. Right: mean elevation. If the two maps look alike, elevation is doing a lot of the work. Remember that part of each box lies outside the county (and, for Busia, in Uganda).

In [ ]:
SIZE = {"nakuru": 110, "busia": 330}  # marker size so squares roughly touch
pix = {}
for s in SITES:
    d = df[s]
    g = d.groupby("pixel_id").agg(lat=("lat", "first"), lon=("lon", "first"),
                                  rain=("precip_mm", "mean"), elev=("elev_mean_m", "first"))
    g["rain_mm_yr"] = g["rain"] * 365.25
    pix[s] = g

fig, ax = plt.subplots(2, 2, figsize=(11, 11))
for i, s in enumerate(SITES):
    g = pix[s]
    for j, (col, label, cmap) in enumerate([("rain_mm_yr", "rain (mm/yr)", "YlGnBu"),
                                            ("elev", "elevation (m)", "terrain")]):
        sc = ax[i, j].scatter(g["lon"], g["lat"], c=g[col], s=SIZE[s], marker="s", cmap=cmap)
        ax[i, j].set_aspect("equal")
        ax[i, j].set_title(f"{s.title()}: {label}")
        ax[i, j].set_xlabel("longitude")
        ax[i, j].set_ylabel("latitude")
        plt.colorbar(sc, ax=ax[i, j], shrink=0.8)
plt.tight_layout()
plt.show()

## 5. Does rain follow elevation?

One dot per pixel. The number printed is the rank (Spearman) correlation: +1 means higher ground is always wetter, 0 means no relationship.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, s in zip(ax, SITES):
    g = pix[s]
    rho = g["rain_mm_yr"].corr(g["elev"], method="spearman")
    a.scatter(g["elev"], g["rain_mm_yr"], s=14, alpha=0.6, color=COLORS[s])
    slope, intercept = np.polyfit(g["elev"], g["rain_mm_yr"], 1)
    xs = np.array([g["elev"].min(), g["elev"].max()])
    a.plot(xs, slope * xs + intercept, color="black", lw=1)
    a.set_title(f"{s.title()}: rank correlation {rho:+.2f}")
    a.set_xlabel("pixel elevation (m)")
    a.set_ylabel("average rainfall (mm/yr)")
plt.tight_layout()
plt.show()

**Take away:** elevation explains a good share of where it rains more, but a lot of scatter remains. Elevation will help the models get the *location* right; it cannot tell them *which day* it rains.

## 6. Rain or no rain? (the zero problem)

Most pixel-days have little or no rain. That matters because a model trained to minimise average error on such data learns to predict "a little rain every day". The table shows what share of pixel-days exceed each rainfall threshold, and the histogram shows how large the rainy-day amounts are (log scale).

In [ ]:
thresholds = [0.0, 0.1, 1, 5, 10, 25, 50]
rows = {}
for s in SITES:
    p = df[s]["precip_mm"]
    rows[s] = {f"> {t} mm": f"{(p > t).mean() * 100:.1f}%" for t in thresholds}
display(pd.DataFrame(rows).T)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = np.logspace(0, np.log10(250), 40)
for a, s in zip(ax, SITES):
    wet = df[s].loc[df[s]["precip_mm"] > 1, "precip_mm"]
    a.hist(wet, bins=bins, color=COLORS[s])
    a.set_xscale("log")
    a.set_yscale("log")
    a.set_title(f"{s.title()}: amounts on days above 1 mm (n={len(wet):,})")
    a.set_xlabel("mm per day (log scale)")
ax[0].set_ylabel("number of pixel-days (log scale)")
plt.tight_layout()
plt.show()

**Take away:** this decides how we model rainfall. If a large share of days are dry and a few days carry big amounts, a two-step approach (first "will it rain?", then "how much?") is worth trying, and we should judge models with rain/no-rain scores as well as RMSE. Write down a wet-day threshold (0.1, 1 or 5 mm). CHIRPS produces many light drizzle days, so 1 mm is a common choice.

## 7. The extreme days

Earlier checks showed single-pixel days of up to 240 mm (Nakuru) and 99 mm (Busia). Is that a real storm that covered many pixels, or a lone spike in one pixel? Real storms show neighbouring pixels and neighbouring days also being wet.

In [ ]:
def pivot(d, col):
    return d.pivot(index="date", columns="pixel_id", values=col)


for s in SITES:
    d = df[s]
    R = pivot(d, "precip_mm")
    top = d.nlargest(8, "precip_mm")[["pixel_id", "date", "precip_mm", "elev_mean_m"]]
    print(f"=== {s.title()} ===")
    print(f"pixel-days >= 100 mm: {int((R.values >= 100).sum())} on {int((R >= 100).any(axis=1).sum())} different days")
    display(top.reset_index(drop=True))
    row = top.iloc[0]
    day = R.loc[row["date"]]
    print(f"Largest event, {row['date']:%Y-%m-%d} at pixel {row['pixel_id']}: "
          f"{int((day > 50).sum())} pixels above 50 mm that day, region average {day.mean():.1f} mm, "
          f"{int((day > 25).sum())} pixels above 25 mm")
    around = R.loc[row["date"] - pd.Timedelta(days=2): row["date"] + pd.Timedelta(days=2), row["pixel_id"]]
    print("same pixel, two days either side (mm):", around.round(1).tolist())
    print()

**Take away:** if the big days are isolated one-pixel spikes, they are likely noise and we may cap or flag them; if they are widespread and last more than a day, they are real events the model should be allowed to learn from. Note your conclusion.

## 8. How predictable is tomorrow? (memory in the data)

Autocorrelation answers: *if I know today, how well do I know the value k days from now?* To make it fair we remove each pixel's typical value for that month first, so the seasonal cycle and the difference between pixels do not inflate the result. A value near 1 means "very predictable from today"; near 0 means "today tells you nothing".

In [ ]:
def anomalies(M):
    return M - M.groupby(M.index.month).transform("mean")


def pooled_acf(M, lags=range(1, 15)):
    X = M.values.astype("float64")
    return [np.corrcoef(X[:-k].ravel(), X[k:].ravel())[0, 1] for k in lags]


lags = list(range(1, 15))
acf = {}
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, (var, label) in zip(ax, [("precip_mm", "Rainfall"), ("rh_pct", "Relative humidity")]):
    for s in SITES:
        A = anomalies(pivot(df[s], var))
        acf[(s, var)] = pooled_acf(A, lags)
        a.plot(lags, acf[(s, var)], marker="o", color=COLORS[s], label=s)
    a.set_title(f"{label}: correlation with the value k days earlier")
    a.set_xlabel("k (days)")
    a.axhline(0, color="black", lw=0.8)
    a.legend()
ax[0].set_ylabel("correlation of anomalies")
plt.tight_layout()
plt.show()

table = pd.DataFrame({f"{s} {v}": [acf[(s, v)][0], acf[(s, v)][2], acf[(s, v)][6]]
                      for s in SITES for v in ["precip_mm", "rh_pct"]},
                     index=["1 day", "3 days", "7 days"]).round(2)
display(table)

**Take away:** this is a realistic preview of forecast skill from the past alone. If rainfall correlation drops close to zero after a day or two, no model built only on past local data can do well at days 5 to 7. That is not a failure of the project: it sets honest expectations, and it is why baselines matter and why your supervisor's idea of fusing in an external forecast is worth pursuing. Humidity should hold up better.

## 9. How local is the information?

For each day we look at how much the values differ **between pixels**. If humidity barely differs from pixel to pixel while rainfall differs a lot, then the local detail in the data comes mainly from CHIRPS rainfall and from elevation, not from the ERA5 weather columns. The second table shows how closely a typical pixel follows its region's daily average (1 means it moves in lock-step with the region).

In [ ]:
rows = {}
for s in SITES:
    d = df[s]
    out = {}
    for var, label in [("precip_mm", "rain"), ("rh_pct", "humidity"), ("t2m_c", "temperature")]:
        M = pivot(d, var)
        out[f"{label}: typical between-pixel spread on a day"] = M.std(axis=1).median()
        out[f"{label}: median pixel-vs-region correlation"] = M.corrwith(M.mean(axis=1)).median()
    rows[s] = out
display(pd.DataFrame(rows).round(2))

**Take away:** ERA5 values are smooth across pixels (they come from a 25 km grid), so the hyper-local signal for rainfall is carried by CHIRPS history and elevation. This is worth stating plainly in your limitations: the model's local skill is bounded by what these inputs can distinguish between neighbouring pixels.

## 10. One-page summary

In [ ]:
rows = {}
for s in SITES:
    d, g = df[s], pix[s]
    rows[s] = {
        "pixels": d["pixel_id"].nunique(),
        "days": d["date"].nunique(),
        "average rain (mm/yr, mean of pixels)": round(g["rain_mm_yr"].mean()),
        "wet days > 1 mm (%)": round((d["precip_mm"] > 1).mean() * 100, 1),
        "mean humidity (%)": round(d["rh_pct"].mean(), 1),
        "mean temperature (C)": round(d["t2m_c"].mean(), 1),
        "elevation range (m)": f"{g['elev'].min():.0f} to {g['elev'].max():.0f}",
        "rain vs elevation (rank corr)": round(g["rain_mm_yr"].corr(g["elev"], method="spearman"), 2),
        "rain autocorr, 1 day": round(acf[(s, "precip_mm")][0], 2),
        "humidity autocorr, 1 day": round(acf[(s, "rh_pct")][0], 2),
    }
display(pd.DataFrame(rows))

## What to write down before moving on

Copy your answers into your project notes. We will use them directly in the next step.

1. **Wet-day threshold:** which value (0.1, 1 or 5 mm) will define "rain"?
2. **Rainfall approach:** is the dry-day share high enough to justify a rain/no-rain stage plus an amount model?
3. **Extreme days:** are the very large values real events (keep) or isolated spikes (cap or flag)?
4. **Horizon expectations:** at what lead time does rainfall predictability from the past fall away? And humidity?
5. **Local signal:** do you agree that local skill will come mainly from CHIRPS history and elevation?
6. **Seasons:** do both sites show the two-season pattern, and does it differ between them?